In [1]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import torchaudio
from datasets import load_dataset, Dataset
from evaluate import load
import librosa
import re

# Load models
models = {
    "mesolotica": WhisperForConditionalGeneration.from_pretrained("mesolitica/malaysian-whisper-small-v3"),
    # "openai": WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3"),
    # "me": WhisperForConditionalGeneration.from_pretrained(r"D:\LingoMalay\Models_Transcribe\API\Kelantan\app\model"),
    # "me": WhisperForConditionalGeneration.from_pretrained(r"D:\LingoMalay\Models_Transcribe\Model\Kedah\whisper-kedahv3\checkpoint-1000")
}
processors = {
    name: WhisperProcessor.from_pretrained(model_name)
    for name, model_name in {
        "mesolotica": "mesolitica/malaysian-whisper-small-v3",
        # "openai": "openai/whisper-large-v3"
        # "me": r"D:\LingoMalay\Models_Transcribe\API\Kelantan\app\model"
        # "me": r"D:\LingoMalay\Models_Transcribe\Model"
    }.items()
}

# Example: list of dictionaries
data = [
    {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kedah\audio_2025-04-19_21-00-49.mp3"}, "sentence": "awat tahabaq mai"},
    {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kedah\audio_2025-04-19_20-46-14.mp3"}, "sentence": "tak celuih laa"},
    {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kedah\audio_2025-07-04_17-23-16.ogg"}, "sentence": "hang pasaipa"},
    {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kedah\audio_2025-07-04_17-48-14.ogg"}, "sentence": "depa nak pi mana tu"},
    {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kedah\audio_2025-07-04_17-15-56.ogg"}, "sentence": "loqlaq la hang ni"},
]

# data = [
#     {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kelantan\audio_2025-06-18_01-55-11.ogg"}, "sentence": "ok lepah demo gi kighi, gi straight ja"},
#     {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kelantan\audio_2025-06-18_01-55-06.ogg"}, "sentence": "dok bakpo ni"},
#     {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kelantan\audio_2025-06-18_01-55-02.ogg"}, "sentence": "macei kite tengok oghe tu buat kijo"},
#     {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kelantan\audio_2025-06-18_01-54-58.ogg"}, "sentence": "nikoh memang banyak kijo pon"},
#     {"audio": {"path": r"D:\LingoMalay\Models_Transcribe\Dataset\Kelantan\audio_2025-06-18_01-54-52.ogg"}, "sentence": "mung nok tubik mano"}
# ]

ds = Dataset.from_list(data)

# cer/wer
wer = load("wer")
cer = load("cer")

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation
    return text.strip()

def transcribe(audio_path, model, processor):
    # Load audio with librosa at 22050 Hz (default for librosa)
    waveform, sample_rate = librosa.load(audio_path, sr=None)  # You can replace 22050 with None for the default sample rate

    # Resample to 16000 Hz (required by Whisper)
    if sample_rate != 16000:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=16000)
    
    # Process the waveform
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt")
    
    # Generate transcription
    with torch.no_grad():
        pred_ids = model.generate(**inputs, forced_decoder_ids=processor.get_decoder_prompt_ids(language="ms", task="transcribe"))
    
    # return processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    output = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    clean_output = output.replace("startoftranscript", "").replace("ms", "").replace("transcribe", "").replace("notimestamps", "").strip()
    return clean_output

# Run benchmark
for name in models:
    predictions = []
    references = []
    for example in ds:
        ref = normalize(example['sentence'])
        audio = example['audio']['path']
        pred = normalize(transcribe(audio, models[name], processors[name]))
        references.append(ref)
        predictions.append(pred)
        print(f"Prediction: {pred}")
        print(f"Ground Truth: {ref}")

    Wscore = wer.compute(predictions=predictions, references=references)
    Cscore = cer.compute(predictions=predictions, references=references)
    print(f"WER for {name}: {Wscore:.4f}")
    print(f"CER for {name}: {Cscore:.4f}")

c:\Users\aqils\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aqils\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\aqils\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\generation\utils.py:869: FutureWarning: You have explicitly specified `forced_decoder_ids`. This functionality has been deprecated and will throw an error in v4.40. Please remove the `forced_decoder_ids

Prediction: awak terhabak max
Ground Truth: awat tahabaq mai
Prediction: tak celih lah
Ground Truth: tak celuih laa
Prediction: hang pasai pa
Ground Truth: hang pasaipa
Prediction: depanak pimana tu
Ground Truth: depa nak pi mana tu
Prediction: lok la le hang ni
Ground Truth: loqlaq la hang ni
WER for mesolotica: 0.7647
CER for mesolotica: 0.1795
